### calculation of JJA 2020-2025 Daily mean temperature versus Snow cover over NA

In [1]:
import os, logging, warnings
from pathlib import Path
import intake
import numpy as np
import xarray as xr
import pyicon as pyic
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from typing import Tuple, List, Optional, Union, Dict

# --- optional map panel (recommended) ---
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.patches import Polygon
import healpy as hp
import healpy
# -------------------------
# environment / dask hygiene
# -------------------------
os.environ.update({
    "HDF5_USE_FILE_LOCKING": "FALSE",
    "OMP_NUM_THREADS": "1",
    "OPENBLAS_NUM_THREADS": "1",
    "MKL_NUM_THREADS": "1",
    "NUMEXPR_NUM_THREADS": "1",
})

warnings.filterwarnings("ignore")
for _n in ["distributed", "dask", "xarray", "fsspec", "numba", "matplotlib"]:
    logging.getLogger(_n).setLevel(logging.ERROR)


----Start loading pyicon.
Loading default parameters from /home/m/m301257/.conda/envs/xianpu/lib/python3.12/site-packages/pyicon/params_default.json.
----Start loading pyicon.
----Pyicon was loaded successfully.
----Pyicon was loaded successfully.


In [2]:
from pathlib import Path
import sys
WAVE_TOOLS_PATH = Path("/work/mh1498/m301257/wave_tools")
sys.path.insert(0, str(WAVE_TOOLS_PATH.parent))
from wave_tools.utils import dataarray_healpix_to_equatorial_latlon
# 网格转换参数
"""

在我给你的function.py文件中,找到
MAXIMUM_LAT_RANGE = 

原本的数值可能是25或者一个小于90的数值，将其改成

MAXIMUM_LAT_RANGE = 90

并放到dataarray_healpix_to_equatorial_latlon这个函数的：

if not HAS_HEALPY:
    raise ImportError("healpy is required for HEALPix operations. Install with: pip install healpy")
    
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
!!!!!!!!!!--> 放到这里: MAXIMUM_LAT_RANGE = 90!!!!!!!!!
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
if minmax_lat > MAXIMUM_LAT_RANGE:
    raise ValueError(f"minmax_lat={minmax_lat} too wide for equatorial analysis.")
    

"""
def dataarray_to_equatorial_latlon_grid(
    dataarray: xr.DataArray, grid_type: str, grid_dict: Optional[dict]
) -> xr.DataArray:
    """转换数据到赤道经纬度网格"""
    if grid_type == "latlon":
        return dataarray
    elif grid_type == "healpix":
        if grid_dict is None:
            raise ValueError("No grid_dict provided for healpix conversion.")
        return dataarray_healpix_to_equatorial_latlon(dataarray, **grid_dict)
    else:
        raise ValueError("Grid type not found.")

In [3]:
def open_icon(key: str) -> xr.Dataset:
    cat = intake.open_catalog(CATALOG_URL)
    entry = cat["ICON"][key]
    try:
        ds = entry(zoom=ZOOM, time=TIME).to_dask()
    except TypeError:
        ds = entry.to_dask()
    return ds


def pick_var(ds: xr.Dataset, candidates, label):
    for v in candidates:
        if v in ds.data_vars:
            return v
    raise KeyError(f"Could not find {label}. Tried: {candidates}\n"
                   f"Available vars (head): {list(ds.data_vars)[:40]}")

In [4]:

order   = 8
nside   = hp.order2nside(order)
npix    = hp.nside2npix(nside)
print(f"Using HEALPix nside={nside}, npix={npix}")
""" 


Healpix 网格参数:

如果你想设置转换后的纬度的最大范围为80度，可以使用以下参数：
"minmax_lat": 80



"""
grid_dict = {"nside": nside, "nest": True, "minmax_lat": 80}

Using HEALPix nside=256, npix=786432


In [5]:
# -------------------------
# config
# -------------------------
CATALOG_URL = "https://data.nextgems-h2020.eu/catalog.yaml"

KEY_A = "ngc3028"
KEY_B = "ngc4008"

ZOOM = 9
TIME = "P1D"   # if your source is not daily, we still enforce daily means below

START_DATE = "2020-01-01"
END_DATE   = "2024-12-31"
DJF_MONTHS = [12, 1, 2] #DJF

# North America box (deg)
LAT_MIN, LAT_MAX = 40.0, 60.0
LON_MIN, LON_MAX = -110.0, -80.0   # west is negative

TIME_CHUNK = 365
CELL_CHUNK = 20000

OUT_DIR = Path(".")
OUT_PDF = OUT_DIR / f"NA_box_DJF_daily_scatter_snow_anom_vs_ts_{KEY_A}_vs_{KEY_B}_2020_2024.pdf"

COL_A = "#1f3b73"  # blue-ish
COL_B = "#c33a2b"  # red-ish

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["STIXGeneral", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "axes.linewidth": 1.6,
    "axes.labelsize": 13,
    "axes.titlesize": 16,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "xtick.major.size": 6,
    "ytick.major.size": 6,
    "xtick.major.width": 1.2,
    "ytick.major.width": 1.2,
    "savefig.bbox": "tight",
    "savefig.transparent": True,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

In [6]:
def regional_daily_DJF_mean(
    ds: xr.Dataset,
    varname: str,
) -> xr.DataArray:
    """
    Extract daily DJF values (no spatial averaging yet).
    Grid-aware, auto regridding.
    """

    da = ds[varname].sel(time=slice(START_DATE, END_DATE))

    if "cell" in da.dims:
        da = da.transpose("time", "cell")

    print(f"{varname}: {da.dims}, {da.shape}")

    # === 核心：统一 grid 入口 ===
    print("Regridding to equatorial lat-lon...")
    da_reg = dataarray_to_equatorial_latlon_grid(da,'healpix', grid_dict)

    da_reg = da_reg.chunk({"time": 30})

    da_daily = da_reg.resample(time="1D").mean()

    # da_djf = da_daily.sel(da_daily.time.dt.season == "DJF")
    da_djf = da_daily.sel(time=da_daily.time.dt.season == "DJF")
    return da_djf.chunk({"time": TIME_CHUNK}).compute()


def area_weighted_mean_latlon(da: xr.DataArray) -> xr.DataArray:
    """Area-weighted mean over lat/lon using cos(lat)."""
    if not {"lat", "lon"}.issubset(da.dims):
        raise ValueError(f"Expected lat/lon dims, got {da.dims}")
    weights_lat = np.cos(np.deg2rad(da["lat"]))
    weights = xr.where(np.isfinite(da), weights_lat, np.nan)
    num = (da * weights).sum(dim=("lat", "lon"), skipna=True)
    den = weights.sum(dim=("lat", "lon"), skipna=True)
    return num / den

def area_weighted_mean_over_cell(da: xr.DataArray, w: xr.DataArray | None):
    if "cell" not in da.dims:
        raise ValueError(f"Expected 'cell' dim, got {da.dims}")
    if w is None:
        return da.mean("cell", skipna=True)

    w2 = w
    if "cell" not in w2.dims:
        w2 = w2.squeeze()
    return (da * w2).sum("cell", skipna=True) / w2.sum("cell", skipna=True)

In [ ]:
def build_for_key(key: str):
    ds = open_icon(key)

    # ---- adjust these if needed ----
    tas_name = pick_var(ds, ["tas", "T_2M", "t2m", "tasmean"], "air temperature (tas or equivalent)")
    ts_name  = pick_var(ds, ["ts", "tsurf", "T_S", "skt", "t_s"], "surface temperature (ts)")
    snow_name = pick_var(
        ds,
        ["hydro_w_snow_box", "hydro_weq_snow_box"],
        "Snow cover anomaly"
    )
    # transform to daily data to the regular 1° lat/lon grid
    
    tas = regional_daily_DJF_mean(ds, tas_name, )
    ts  = regional_daily_DJF_mean(ds, ts_name, )
    snow = regional_daily_DJF_mean(ds, snow_name, )
    
    # subset NA box on regular grid
    tas = tas.sel(lat=slice(LAT_MIN, LAT_MAX), lon=slice(LON_MIN, LON_MAX))
    ts  = ts.sel(lat=slice(LAT_MIN, LAT_MAX), lon=slice(LON_MIN, LON_MAX))
    snow = snow.sel(lat=slice(LAT_MIN, LAT_MAX), lon=slice(LON_MIN, LON_MAX))

    tas = area_weighted_mean_latlon(tas)
    ts  = area_weighted_mean_latlon(ts)
    snow = area_weighted_mean_latlon(snow)

    snow_anom = snow - snow.mean("time")

    tas, ts, snow_anom = xr.align(tas, ts, snow_anom, join="inner")
    dt = ts
    
    return dt, snow_anom, (tas_name, ts_name, snow_name)


In [8]:
# # -------------------------
# # compute
# # -------------------------
dt_a, snow_anom_a, names_a = build_for_key(KEY_A)



tas: ('time', 'cell'), (1807, 3145728)
Regridding to equatorial lat-lon...
DEBUG: Total pixels: 786432
DEBUG: Lat range: -89.82 to 89.82
DEBUG: Lon range: 0.00 to 359.82
DEBUG: Unique latitudes in range: 915
DEBUG: Reference latitude: -41.81, with 1024 longitude points
  Lat -79.9363: 220 points, target: 1024 points
    -> Interpolated to 1024 points
  Lat -79.7528: 224 points, target: 1024 points
    -> Interpolated to 1024 points
  Lat -79.5693: 228 points, target: 1024 points
    -> Interpolated to 1024 points
  Lat 79.9363: 220 points, target: 1024 points
ts: ('time', 'cell'), (1807, 3145728)
Regridding to equatorial lat-lon...
DEBUG: Total pixels: 786432
DEBUG: Lat range: -89.82 to 89.82
DEBUG: Lon range: 0.00 to 359.82
DEBUG: Unique latitudes in range: 915
DEBUG: Reference latitude: -41.81, with 1024 longitude points
  Lat -79.9363: 220 points, target: 1024 points
    -> Interpolated to 1024 points
  Lat -79.7528: 224 points, target: 1024 points
    -> Interpolated to 1024 points

In [11]:
dt_b, snow_anom_b, names_b = build_for_key(KEY_B)

tas: ('time', 'cell'), (1826, 3145728)
Regridding to equatorial lat-lon...
DEBUG: Total pixels: 786432
DEBUG: Lat range: -89.82 to 89.82
DEBUG: Lon range: 0.00 to 359.82
DEBUG: Unique latitudes in range: 915
DEBUG: Reference latitude: -41.81, with 1024 longitude points
  Lat -79.9363: 220 points, target: 1024 points
    -> Interpolated to 1024 points
  Lat -79.7528: 224 points, target: 1024 points
    -> Interpolated to 1024 points
  Lat -79.5693: 228 points, target: 1024 points
    -> Interpolated to 1024 points
  Lat 79.9363: 220 points, target: 1024 points
ts: ('time', 'cell'), (1826, 3145728)
Regridding to equatorial lat-lon...
DEBUG: Total pixels: 786432
DEBUG: Lat range: -89.82 to 89.82
DEBUG: Lon range: 0.00 to 359.82
DEBUG: Unique latitudes in range: 915
DEBUG: Reference latitude: -41.81, with 1024 longitude points
  Lat -79.9363: 220 points, target: 1024 points
    -> Interpolated to 1024 points
  Lat -79.7528: 224 points, target: 1024 points
    -> Interpolated to 1024 points

In [12]:
def fit_line(x: xr.DataArray, y: xr.DataArray):
    """
    Fit y = a*x + b using finite points only.
    Returns (a, b).
    """
    xv = np.asarray(x.values).ravel()
    yv = np.asarray(y.values).ravel()
    m = np.isfinite(xv) & np.isfinite(yv)
    if m.sum() < 3:
        return np.nan, np.nan
    a, b = np.polyfit(xv[m], yv[m], 1)
    return float(a), float(b)
a_a, b_a = fit_line(dt_a, snow_anom_a)
a_b, b_b = fit_line(dt_b, snow_anom_b)

In [14]:
a_a

nan

In [1]:
# -------------------------
# plot: map + scatter
# -------------------------
fig = plt.figure(figsize=(14.5, 5.2), constrained_layout=True)
gs = fig.add_gridspec(1, 2, width_ratios=[1.05, 1.25])

# (1) map panel
axm = fig.add_subplot(gs[0, 0], projection=ccrs.Robinson())
axm.set_title("North America box", pad=10)
axm.add_feature(cfeature.COASTLINE, linewidth=1.2)
axm.add_feature(cfeature.BORDERS, linewidth=0.6, alpha=0.6)
axm.gridlines(linewidth=0.6, alpha=0.4, linestyle="--")

# draw the lat-lon box (PlateCarree coordinates)
box_coords = np.array([
    [LON_MIN, LAT_MIN],
    [LON_MAX, LAT_MIN],
    [LON_MAX, LAT_MAX],
    [LON_MIN, LAT_MAX],
])
poly = Polygon(box_coords, closed=True, fill=False, edgecolor="red", linewidth=2.5,
               transform=ccrs.PlateCarree())
axm.add_patch(poly)

# (2) scatter panel
axs = fig.add_subplot(gs[0, 1])
axs.set_title("North America box mean daily values (JJA 2020–2025)")

# scatter (open circles)
axs.scatter(dt_b.values, snow_anom_b.values, s=55, facecolors="none", edgecolors=COL_B, alpha=0.45, linewidths=1.3, label=KEY_B)
axs.scatter(dt_a.values, snow_anom_a.values, s=55, facecolors="none", edgecolors=COL_A, alpha=0.45, linewidths=1.3, label=KEY_A)

# regression lines
x_min = np.nanmin([dt_a.min().item(), dt_b.min().item()])
x_max = np.nanmax([dt_a.max().item(), dt_b.max().item()])
xx = np.linspace(x_min, x_max, 200)

if np.isfinite(a_b) and np.isfinite(b_b):
    axs.plot(xx, a_b * xx + b_b, linestyle="--", linewidth=2.5, color=COL_B)
if np.isfinite(a_a) and np.isfinite(b_a):
    axs.plot(xx, a_a * xx + b_a, linestyle="--", linewidth=2.5, color=COL_A)

# labels
axs.set_xlabel("Air temperature − surface temperature (K)")  # change to “12m air temp” if that’s your actual variable
axs.set_ylabel("Snow cover (m)")
axs.grid(True, alpha=0.35)
axs.spines["top"].set_visible(False)
axs.spines["right"].set_visible(False)

# annotation: slope/intercept like your example
txt = (
    f"{KEY_B}: snow_anom = {a_b:5.2f} m/K  {b_b:+6.1f} m\n"
    f"{KEY_A}: snow_anom = {a_a:5.2f} m/K  {b_a:+6.1f} m"
)
t = axs.text(0.03, 0.95, txt, transform=axs.transAxes, va="top", ha="left", fontsize=13)
t.set_path_effects([pe.Stroke(linewidth=4, foreground="white", alpha=0.7), pe.Normal()])

leg = axs.legend(frameon=True, loc="lower right", fontsize=14)
leg.get_frame().set_alpha(0.9)

# OUT_DIR.mkdir(parents=True, exist_ok=True)
# fig.savefig(OUT_PDF, format="pdf")
# fig.savefig(OUT_PDF.with_suffix(".png"), format="png")
plt.show()

print("Variable mapping A:", dict(zip(["tas","ts","snow_anom"], names_a)))
print("Variable mapping B:", dict(zip(["tas","ts","snow_anom"], names_b)))
# print(f"Saved: {OUT_PDF}")

NameError: name 'plt' is not defined